# 03 — Model training and versioning\n\nThis notebook demonstrates the training pipeline, cross-validation, model registry, and champion/challenger comparison logic.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_DIR)

from src.data_loader import load_training_data
from src.model import (
    ModelTrainer, ModelEvaluator, ModelRegistry,
    FeatureEngineer, compare_models, MetricsLogger,
)

In [ ]:
data_dir = os.path.join(PROJECT_DIR, "data")
train_df, test_df = load_training_data(data_dir)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

## Train model with cross-validation

In [ ]:
trainer = ModelTrainer()
cv_results = trainer.cross_validate(train_df, cv=5)
print(f"5-fold CV AUC: {cv_results['mean_auc']:.4f} +/- {cv_results['std_auc']:.4f}")

In [ ]:
# Fit on full training set
trainer.fit(train_df)
print("Model trained successfully")

## Evaluate on holdout set

In [ ]:
metrics = ModelEvaluator.evaluate(trainer, test_df)

print("Holdout metrics:")
for k, v in metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

In [ ]:
# ROC curve
from sklearn.metrics import roc_curve, auc

y_true = test_df["Churn"].values
y_prob = trainer.predict_proba(test_df)
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color="#E8C230", lw=2, label=f"AUC = {roc_auc:.4f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve — holdout set")
ax.legend()
plt.tight_layout()
plt.show()

## Model registry\n\nSave the trained model to the local registry with timestamp-based versioning.

In [ ]:
artifacts_dir = os.path.join(PROJECT_DIR, "artifacts")
registry = ModelRegistry(artifacts_dir)

# Save model version A
version_a = registry.save_model(trainer, metrics, tag="v1_baseline")
registry.promote_to_production(version_a)
print(f"Saved and promoted: {version_a}")

## Champion vs challenger comparison\n\nTrain a challenger model with different hyperparameters and compare it against the champion.

In [ ]:
# Train challenger with different params
challenger_params = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "random_state": 42,
}
challenger = ModelTrainer(model_params=challenger_params)
challenger.fit(train_df)
challenger_metrics = ModelEvaluator.evaluate(challenger, test_df)

print("Challenger metrics:")
for k, v in challenger_metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

In [ ]:
# A/B comparison
comparison = compare_models(metrics, challenger_metrics)
print(f"Primary metric: {comparison['primary_metric']}")
print(f"Champion:       {comparison['champion_metric']:.4f}")
print(f"Challenger:     {comparison['challenger_metric']:.4f}")
print(f"Difference:     {comparison['difference']:+.4f}")
print(f"Threshold:      {comparison['threshold']}")
print(f"Promote?        {comparison['promote_challenger']}")

In [ ]:
# Visual comparison
metric_names = list(metrics.keys())
champion_vals = [metrics[m] for m in metric_names]
challenger_vals = [challenger_metrics[m] for m in metric_names]

x = np.arange(len(metric_names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, champion_vals, w, label="Champion (v1)", color="#3B6FD4")
ax.bar(x + w/2, challenger_vals, w, label="Challenger (v2)", color="#E8C230")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel("Score")
ax.set_title("Champion vs challenger — all metrics")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Save challenger to registry
version_b = registry.save_model(challenger, challenger_metrics, tag="v2_challenger")
print(f"Saved: {version_b}")

# List all versions
versions = registry.list_versions()
for v in versions:
    m = v.get("metrics", {})
    prod = " (production)" if v["version"] == registry.get_production_version() else ""
    print(f"  {v['version']}: AUC={m.get('roc_auc', 0):.4f}{prod}")

## Feature importance

In [ ]:
# Extract feature importances from the GBM inside the pipeline
gbm = trainer.pipeline.named_steps["classifier"]
preprocessor = trainer.pipeline.named_steps["preprocessor"]

num_names = list(preprocessor.transformers_[0][2])
cat_names = list(preprocessor.transformers_[1][1].get_feature_names_out())
all_names = num_names + cat_names

importances = gbm.feature_importances_
idx = np.argsort(importances)[-15:]  # top 15

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh([all_names[i] for i in idx], importances[idx], color="#3B6FD4")
ax.set_xlabel("Feature importance")
ax.set_title("Top 15 features — GradientBoosting classifier")
plt.tight_layout()
plt.show()

## Summary\n\n- GradientBoosting classifier achieves strong holdout AUC with stable cross-validation scores\n- Model registry stores versioned artifacts with full metadata (parameters, metrics, timestamps)\n- Champion/challenger comparison automates promotion decisions based on configurable thresholds\n- The production pointer file tracks which version is currently deployed